# DTAAD: Dual TCN-Attention Networks for Anomaly Detection
**논문**: Knowledge-Based Systems 295 (2024) 111849

### ⚠️ 시작 전: 런타임 → 런타임 유형 변경 → GPU (T4) 선택 후 순서대로 실행

## 1단계: 저장소 클론 + 패키지 설치

In [ ]:
!git clone https://github.com/Yu-Lingrui/DTAAD.git
%cd DTAAD
!pip install -q pandas tqdm matplotlib numpy scikit-learn seaborn SciencePlots xlrd==1.2.0
print('✅ 완료')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 2단계: 전체 패치 (9가지 버그 한 번에 수정)

| 패치 | 대상 | 문제 |
|------|------|------|
| 1 | `src/parser.py` | Jupyter 커널 인자 충돌 |
| 2 | `src/models.py` | DGL 설치 불가 |
| 3 | `src/plotting.py` | SciencePlots 스타일 없음 |
| 4 | `src/dlutils.py` | PyTorch 최신 버전 `is_causal` 충돌 |
| 5 | `main.py` | pandas `df.append()` 삭제됨 |
| 6 | `main.py` | training 단계 permute 차원 오류 |
| 7 | `main.py` | `plot_attention` 속성 없는 모델에서 오류 |
| 8 | `src/pot.py` | `while True` 무한루프 |
| 9 | `main.py` | testing 단계 permute 차원 오류 |

In [ ]:
# ── 패치 1: parser.py - Jupyter 커널 인자 충돌 ──────────────────────
with open('src/parser.py', 'r') as f:
    c = f.read()
if 'args = parser.parse_args()' in c:
    c = c.replace('args = parser.parse_args()',
                  'args, _ = parser.parse_known_args()')
    open('src/parser.py', 'w').write(c)
    print('✅ 패치 1 완료: parser.py')
else:
    print('ℹ️  패치 1: 이미 적용됨')

# ── 패치 2: models.py - DGL import 오류 ─────────────────────────────
with open('src/models.py', 'r') as f:
    c = f.read()
old = 'import dgl.nn\nfrom dgl.nn.pytorch import GATConv'
new = ('try:\n'
       '    import dgl.nn\n'
       '    from dgl.nn.pytorch import GATConv\n'
       'except Exception:\n'
       '    GATConv = None')
if old in c:
    open('src/models.py', 'w').write(c.replace(old, new))
    print('✅ 패치 2 완료: models.py')
else:
    print('ℹ️  패치 2: 이미 적용됨')

# ── 패치 3: plotting.py - SciencePlots 스타일 없음 ───────────────────
with open('src/plotting.py', 'r') as f:
    c = f.read()
old = "plt.style.use(['science', 'ieee'])"
new = ("try:\n"
       "    plt.style.use(['science', 'ieee'])\n"
       "except OSError:\n"
       "    plt.style.use('default')")
if old in c:
    open('src/plotting.py', 'w').write(c.replace(old, new))
    print('✅ 패치 3 완료: plotting.py')
else:
    print('ℹ️  패치 3: 이미 적용됨')

# ── 패치 4: dlutils.py - PyTorch 최신 버전 is_causal 충돌 ───────────
with open('src/dlutils.py', 'r') as f:
    c = f.read()
old_enc = ('def forward(self, src, src_mask=None, src_key_padding_mask=None):\n'
           '        src2 = self.self_attn(src, src, src)[0]')
new_enc = ('def forward(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):\n'
           '        src2 = self.self_attn(src, src, src)[0]')
old_dec = ('def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None,\n'
           '                memory_key_padding_mask=None):')
new_dec = ('def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None,\n'
           '                memory_key_padding_mask=None, **kwargs):')
changed = False
if old_enc in c:
    c = c.replace(old_enc, new_enc); changed = True
if old_dec in c:
    c = c.replace(old_dec, new_dec); changed = True
if changed:
    open('src/dlutils.py', 'w').write(c)
    print('✅ 패치 4 완료: dlutils.py')
else:
    print('ℹ️  패치 4: 이미 적용됨')

# ── 패치 5: main.py - pandas df.append() 삭제 문제 ──────────────────
with open('main.py', 'r') as f:
    c = f.read()
old = '        df = df.append(result, ignore_index=True)'
new = '        df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)'
if old in c:
    c = c.replace(old, new)
    open('main.py', 'w').write(c)
    print('✅ 패치 5 완료: df.append → pd.concat')
else:
    print('ℹ️  패치 5: 이미 적용됨')

# ── 패치 6: main.py - training 단계 permute 차원 오류 ────────────────
with open('main.py', 'r') as f:
    c = f.read()
old2 = '                l1 = _lambda * l(z[0].permute(1, 0, 2), elem) + (1 - _lambda) * l(z[1].permute(1, 0, 2),elem)'
new2 = ('                z0 = z[0].permute(1,0,2) if z[0].dim()==3 else z[0].unsqueeze(0)\n'
        '                z1 = z[1].permute(1,0,2) if z[1].dim()==3 else z[1].unsqueeze(0)\n'
        '                l1 = _lambda * l(z0, elem) + (1 - _lambda) * l(z1, elem)')
if old2 in c:
    c = c.replace(old2, new2)
    open('main.py', 'w').write(c)
    print('✅ 패치 6 완료: training permute 차원 안전 처리')
else:
    print('ℹ️  패치 6: 이미 적용됨')

# ── 패치 7: main.py - plot_attention 모델에 encoder 없으면 스킵 ───────
with open('main.py', 'r') as f:
    c = f.read()
old3 = ("    ### Plot attention\n"
        "    if not args.test:\n"
        "        if 'DTAAD' in model.name:\n"
        "            plot_attention(model, 1, f'{args.model}_{args.dataset}')")
new3 = ("    ### Plot attention\n"
        "    if not args.test:\n"
        "        if 'DTAAD' in model.name and hasattr(model, 'transformer_encoder1'):\n"
        "            plot_attention(model, 1, f'{args.model}_{args.dataset}')")
if old3 in c:
    c = c.replace(old3, new3)
    open('main.py', 'w').write(c)
    print('✅ 패치 7 완료: plot_attention hasattr 가드 추가')
else:
    print('ℹ️  패치 7: 이미 적용됨')

# ── 패치 8: src/pot.py - while True 무한루프 → 최대 1000회로 제한 ────
with open('src/pot.py', 'r') as f:
    c = f.read()
old = '''    lms = lm[0]
    while True:
        try:
            s = SPOT(q)  # SPOT object
            s.fit(init_score, score)  # data import
            s.initialize(level=lms, min_extrema=False, verbose=False)  # initialization step
        except: lms = lms * 0.999
        else: break'''
new = '''    lms = lm[0]
    max_iter = 1000
    for _ in range(max_iter):
        try:
            s = SPOT(q)  # SPOT object
            s.fit(init_score, score)  # data import
            s.initialize(level=lms, min_extrema=False, verbose=False)  # initialization step
            break
        except:
            lms = lms * 0.999
    else:
        pot_th = np.percentile(score, 100 * lm[0]) * lm[1]
        pred, p_latency = adjust_predicts(score, label, pot_th, calc_latency=True)
        p_t = calc_point2point(pred, label)
        return {
            "f1": p_t[0], "precision": p_t[1], "recall": p_t[2],
            "TP": p_t[3], "TN": p_t[4], "FP": p_t[5], "FN": p_t[6],
            "ROC/AUC": p_t[7], "threshold": pot_th
        }, np.array(pred)'''
if old in c:
    c = c.replace(old, new)
    open('src/pot.py', 'w').write(c)
    print('✅ 패치 8 완료: pot.py 무한루프 제한')
else:
    print('ℹ️  패치 8: 이미 적용됨')

# ── 패치 9: main.py - testing 단계 permute 차원 오류 ─────────────────
with open('main.py', 'r') as f:
    c = f.read()
old = '            z = z[1].permute(1, 0, 2)'
new = '            z = z[1].permute(1,0,2) if z[1].dim()==3 else z[1].unsqueeze(0)'
if old in c:
    c = c.replace(old, new)
    open('main.py', 'w').write(c)
    print('✅ 패치 9 완료: testing permute 차원 안전 처리')
else:
    print('ℹ️  패치 9: 이미 적용됨')

In [ ]:
# 패치 검증 - import 테스트
import sys
sys.path.insert(0, '.')
from src.parser import args
print(f'✅ parser 정상: dataset={args.dataset}, model={args.model}')
from src.models import DTAAD
print('✅ DTAAD import 성공!')

## 3단계: 데이터 확인
SMAP & MSL 데이터는 저장소에 이미 포함되어 있습니다 (`data/SMAP_MSL/`)

In [ ]:
import os, pandas as pd
print('train:', len(os.listdir('data/SMAP_MSL/train')), '개')
print('test: ', len(os.listdir('data/SMAP_MSL/test')),  '개')
df = pd.read_csv('data/SMAP_MSL/labeled_anomalies.csv')
print('SMAP:', len(df[df['spacecraft']=='SMAP']), '채널')
print('MSL: ', len(df[df['spacecraft']=='MSL']),  '채널')

## 4단계: 데이터 전처리

In [ ]:
!python3 preprocess.py SMAP MSL
print('\n✅ 전처리 완료')

In [ ]:
# 전처리 결과는 processed/ 폴더에 저장됨
for ds in ['SMAP', 'MSL']:
    files = os.listdir(f'processed/{ds}')
    print(f'{ds}: {len(files)}개 파일 (샘플: {files[:2]})')

## 5단계: DTAAD 학습 및 평가
논문 기대치: SMAP F1=**0.9023** / MSL F1=**0.9495**

In [ ]:
!python3 main.py --model DTAAD --dataset SMAP --retrain

In [ ]:
!python3 main.py --model DTAAD --dataset MSL --retrain

## 6단계: 20% 데이터 실험 (논문 Table 4)

In [ ]:
!python3 main.py --model DTAAD --dataset SMAP --retrain --less

## 7단계: Ablation Study (논문 Table 7)

In [ ]:
!python3 main.py --model DTAAD_Tcn_Local   --dataset SMAP --retrain --less
!python3 main.py --model DTAAD_Tcn_Global  --dataset SMAP --retrain --less
!python3 main.py --model DTAAD_Callback    --dataset SMAP --retrain --less
!python3 main.py --model DTAAD_Transformer --dataset SMAP --retrain --less

## 8단계: 논문 결과 시각화

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

results = {
    'Dataset':   ['NAB',   'UCR',   'MBA',   'SMAP',  'MSL',   'SWaT',  'WADI',  'SMD',   'MSDS'],
    'F1':        [0.9412,  0.9407,  0.9800,  0.9023,  0.9495,  0.8101,  0.5455,  0.9147,  0.8905],
    'AUC':       [0.9996,  0.9988,  0.9896,  0.9911,  0.9918,  0.8462,  0.6950,  0.9892,  0.9013],
}
df = pd.DataFrame(results).set_index('Dataset')
print(df.to_string())
print(f'\n평균 F1={df["F1"].mean():.4f}, 평균 AUC={df["AUC"].mean():.4f}')

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(df))
ax.bar([i-.2 for i in x], df['F1'],  width=.35, label='F1',  color='steelblue')
ax.bar([i+.2 for i in x], df['AUC'], width=.35, label='AUC', color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(df.index)
ax.set_ylim(0.4, 1.05); ax.set_ylabel('Score')
ax.set_title('DTAAD Performance (Paper Table 3)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('dtaad_results.png', dpi=150); plt.show()